# 3. Full Ukrainian generation

Build 1,006 retained cohort profiles and add feedback item `WB-1201`, giving
**1,007 generation inputs**. Use the selected Lapa model to generate a stereotype
(S), a counter-stereotype (C) from S, and an unrelated sentence (U) from S and C.

This walkthrough shows the taxonomy, profile selection, prompts and execution.
The default prepares requests without calling the model.


In [1]:
import json
from pathlib import Path

from pandas import DataFrame, set_option

from warbias_kit.dataset_workflow import create_workspace, find_project, load_inputs
from warbias_kit.notebook_display import display
from warbias_kit.warbias.cohort import load_cohort

project = find_project()
inputs = load_inputs(project)
cohort = load_cohort(project, inputs)
matrix = cohort["matrix"]
protocol = json.loads((project / "protocol/construction.json").read_text())
cohort_protocol = json.loads((project / "protocol/cohort.json").read_text())
work = create_workspace("03_full_ukrainian_generation")
set_option("display.max_colwidth", None)
set_option("display.max_rows", 150)

## Taxonomy and sources

The [bilingual taxonomy](../protocol/inputs/cohort_taxonomy.csv) lists 120 general
cases in 27 categories, with claims, contexts and sources. Ukrainian claims drive
generation; English claims provide their translations. The membership flag marks
119 cases from the original annotation cohort and one feedback addition.

Cases whose claims depend on a named country, institution, legal status, language
or regional identity tied to a source are excluded. Sources record provenance;
they do not measure stereotype prevalence.

`source_urls` and `source_titles` combine all references. `evidence_ids` and
`supporting_evidence_ids` distinguish claim sources from supporting references;
`category_evidence_ids` records category feedback. `source_adaptation` marks cases
built from topics or subtypes in cited sources.

The [evidence mapping](../protocol/inputs/cohort_evidence.json) links eight
adaptations to source sections and records pending checks for two claims and
six subgroup variants. Checks apply to every profile of a case or to the listed
item IDs, as specified. Generated text has a separate human review status.
Missing checks do not imply verified support.


In [2]:
taxonomy = DataFrame(cohort["taxonomy_catalog"])
taxonomy["triplet_count"] = taxonomy["triplet_count"].astype(int)
category_table = (
    taxonomy.groupby(["target_group", "category_en", "category_uk"], sort=False)
    .agg(cases=("case_id", "size"), triplets=("triplet_count", "sum"))
    .reset_index()
)
display(category_table)
display(
    taxonomy[
        [
            "public_case_id",
            "category_en",
            "claim_en",
            "case_scope",
            "evidence_ids",
            "supporting_evidence_ids",
            "triplet_count",
            "in_original_annotation_cohort",
        ]
    ].head(8)
)
display(
    DataFrame(cohort["sources"])[["evidence_id", "title", "source_type", "url"]].fillna("").head(8)
)
evidence = DataFrame(cohort["evidence"])
display(
    evidence[
        [
            "public_case_id",
            "evidence_basis",
            "anchor_status",
            "scope",
            "item_ids",
            "evidence_ids",
            "source_urls",
            "source_locator",
            "note",
        ]
    ].head(8)
)

,target_group,category_en,category_uk,cases,triplets
0,idp,Social assistance and resource allocation,Соціальна допомога та розподіл ресурсів,5,40
1,idp,Civic and political participation,Громадянська і політична участь,3,32
2,idp,Social integration and belonging,Соціальна інтеграція і належність,4,35
3,idp,Crime and antisocial behaviour,Злочинність і антисоціальна поведінка,5,36
4,idp,"Employment, competence and reliability at work",Працевлаштування компетентність і трудова надійність,5,52
5,idp,Housing and tenant reliability,Житло і надійність орендаря,7,72
6,idp,"Loyalty, security and collective responsibility",Лояльність безпека і колективна відповідальність,5,24
7,idp,Mental health and universal trauma,Психічне здоров’я й універсальна травмованість,5,40
8,idp,"Passivity, victimization and agency",Пасивність віктимізація та агентність,4,43
9,idp,"Return, local integration and freedom of choice",Повернення місцева інтеграція і право вибору,3,35


,public_case_id,category_en,claim_en,case_scope,evidence_ids,supporting_evidence_ids,triplet_count,in_original_annotation_cohort
0,WB-C001,Social assistance and resource allocation,[TARGET] constantly depend on assistance.,general,src_idp_protection_2025,,7,true
1,WB-C002,Social assistance and resource allocation,[TARGET] do not deserve the assistance they receive.,general,src_idp_protection_2025,,8,true
2,WB-C003,Social assistance and resource allocation,[TARGET] cheat the system to receive benefits.,general,src_idp_protection_2025,,8,true
3,WB-C004,Social assistance and resource allocation,[TARGET] receive an unfair advantage in the allocation of assistance.,general,src_idp_protection_2025,,10,true
4,WB-C005,Social assistance and resource allocation,[TARGET] take social resources away from local residents.,general,src_idp_protection_2025,,7,true
5,WB-C006,Civic and political participation,[TARGET] should not influence community decisions.,general,src_idp_protection_2025,,10,true
6,WB-C007,Civic and political participation,[TARGET] should not influence how the local budget and services are allocated.,general,src_idp_protection_2025,,11,true
7,WB-C008,Civic and political participation,[TARGET] cannot represent the interests of the local community.,general,src_idp_protection_2025,,11,true


,evidence_id,title,source_type,url
0,src_idp_housing,Оренда житла в Україні: сучасний стан і виклики,published_reference,https://cedos.org.ua/researches/orenda-zhytla-v-ukrayini-suchasnyj-stan-i-vykly/
1,src_idp_protection_2025,Опитування з питань захисту №1,published_reference,https://www.unhcr.org/ua/sites/ua/files/2025-07/Protection%20Survey%201%20-%20Final%20-UKR.pdf
2,src_idp_human_impact,Оцінка людського впливу війни в Україні,published_reference,https://ukraine.un.org/sites/default/files/2023-06/undp-ua-hia-p_1.pdf
3,src_idp_media_agency,Refugees IDPs or Just Ukrainians? Local Online Media and Perceptions of Donbas Internally Displaced Persons,published_reference,https://ceemr.uw.edu.pl/vol-12-no-1-2023/special-section/refugees-idps-or-just-ukrainians-local-online-media-and-perceptions
4,src_idp_mental_health,Психічне здоров’я та психосоціальна підтримка,published_reference,https://www.unhcr.org/what-we-do/protect-human-rights/public-health/mental-health-and-psychosocial-support
5,src_idp_action_agenda,План дій Генерального секретаря з питань внутрішнього переміщення,published_reference,https://www.un.org/en/content/action-agenda-on-internal-displacement/index.shtml
6,src_idp_intentions,Опитування намірів переміщених людей,published_reference,https://www.unhcr.org/what-we-do/build-better-futures/solutions/intentions-surveys
7,src_vet_stigma_2025,Стигматизація ветеранів і ветеранок: ситуація в Україні та досвід США,published_reference,https://veteranfund.com.ua/wp-content/uploads/2025/04/Styhmatyzatsiia-veteraniv-i-veteranok.pdf


,public_case_id,evidence_basis,anchor_status,scope,item_ids,evidence_ids,source_urls,source_locator,note
0,WB-C006,subgroup_transfer,new_target_pending_review,items,[WB-0778],[src_idp_protection_2025],[https://www.unhcr.org/ua/sites/ua/files/2025-07/Protection%20Survey%201%20-%20Final%20-UKR.pdf],"Evidence register, row 8.",This check applies only to the listed profiles with displacement_origin=occupied_territories. The cited sources concern IDPs more broadly; support for this specific subgroup remains unverified.
1,WB-C012,subgroup_transfer,new_target_pending_review,items,[WB-0995],[src_idp_protection_2025],[https://www.unhcr.org/ua/sites/ua/files/2025-07/Protection%20Survey%201%20-%20Final%20-UKR.pdf],"Evidence register, row 6.",This check applies only to the listed profiles with displacement_origin=occupied_territories. The cited sources concern IDPs more broadly; support for this specific subgroup remains unverified.
2,WB-C019,subgroup_transfer,new_target_pending_review,items,[WB-0438],[src_idp_protection_2025],[https://www.unhcr.org/ua/sites/ua/files/2025-07/Protection%20Survey%201%20-%20Final%20-UKR.pdf],"Evidence register, row 5.",This check applies only to the listed profiles with displacement_origin=occupied_territories. The cited sources concern IDPs more broadly; support for this specific subgroup remains unverified.
3,WB-C027,subgroup_transfer,new_target_pending_review,items,[WB-0387],[src_idp_housing],[https://cedos.org.ua/researches/orenda-zhytla-v-ukrayini-suchasnyj-stan-i-vykly/],"Evidence register, row 4.",This check applies only to the listed profiles with displacement_origin=occupied_territories. The cited sources concern IDPs more broadly; support for this specific subgroup remains unverified.
4,WB-C040,subgroup_transfer,new_target_pending_review,items,[WB-0741],[src_idp_media_agency],[https://ceemr.uw.edu.pl/vol-12-no-1-2023/special-section/refugees-idps-or-just-ukrainians-local-online-media-and-perceptions],"Evidence register, row 7.",This check applies only to the listed profiles with displacement_origin=occupied_territories. The cited sources concern IDPs more broadly; support for this specific subgroup remains unverified.
5,WB-C043,subgroup_transfer,new_target_pending_review,items,[WB-0411],"[src_idp_action_agenda, src_idp_intentions]","[https://www.un.org/en/content/action-agenda-on-internal-displacement/index.shtml, https://www.unhcr.org/what-we-do/build-better-futures/solutions/intentions-surveys]","Evidence register, row 9.",This check applies only to the listed profiles with displacement_origin=occupied_territories. The cited sources concern IDPs more broadly; support for this specific subgroup remains unverified.
6,WB-C111,adapted_hypothesis,pending_anchor_review,case,[],"[src_vet_help_stigma, src_vet_stigma_2025]","[https://doi.org/10.38014/ehs-ss.2020.4.01, https://veteranfund.com.ua/wp-content/uploads/2025/04/Styhmatyzatsiia-veteraniv-i-veteranok.pdf]","Evidence register, row 3.",The sources concern stigma around psychological help. The exact help-seeking-as-weakness claim remains an adapted hypothesis with its anchor pending review.
7,WB-C112,adapted_hypothesis,pending_anchor_review,case,[],[src_vet_stigma_2025],[https://veteranfund.com.ua/wp-content/uploads/2025/04/Styhmatyzatsiia-veteraniv-i-veteranok.pdf],"Evidence register, row 2.",The source concerns expectations about veterans. The exact pressure-to-serve-again claim remains an adapted hypothesis with its anchor pending review.


## Select the base and profile extensions

The recorded pool has 127 cases and 1,511 candidate profiles. Each profile allows
three framings: no speaker and two case-specific speakers, giving 4,533 candidates.
The selector reserves one unmodified profile without a speaker and one with a
speaker per case. It then selects complete case–attribute blocks up to 1,000 rows.
It prioritizes cases with fewer blocks, then attributes with fewer selected rows;
SHA256 ties use seed `20260909`. A recorded 16-for-16 substitution preserves the
1,000-row budget. A separate seeded order assigns the WB item IDs.

The 200 extensions form comparison panels: 50 IDP gender variants,
6 veteran gender controls, 51 age variants, 51 occupation variants and
42 gender × age variants.


In [3]:
from copy import deepcopy

from warbias_kit.warbias.allocation import reproduce_base_allocation, validate_recorded_allocation

allocation_design = json.loads((project / "protocol/allocation_design.json").read_text())
replay_design = deepcopy(allocation_design)
replay_design["base_selection"]["occupation_reference_by_case"] = cohort_protocol["selection"][
    "occupation_reference_by_case"
]
substitutions = json.loads(
    (project / allocation_design["base_selection"]["substitutions"]).read_text()
)
reproduced = reproduce_base_allocation(
    inputs["construction_cases"], inputs["candidate_profiles"], replay_design, substitutions
)
validate_recorded_allocation(
    inputs["construction_cases"],
    inputs["candidate_profiles"],
    inputs["profile_design"],
    replay_design,
    substitutions,
)
display(
    {
        "candidate_profiles": len(inputs["candidate_profiles"]),
        "reproduced_base_inputs": len(reproduced),
        "recorded_extensions": len(inputs["profile_design"]) - len(reproduced),
        "base_IDs_and_profiles_match": True,
    }
)
display(
    DataFrame(allocation_design["extensions"]["groups"])[
        ["kind", "case_count", "variants_per_case", "variant_count"]
    ]
)

{'candidate_profiles': 1511,
 'reproduced_base_inputs': 1000,
 'recorded_extensions': 200,
 'base_IDs_and_profiles_match': True}

,kind,case_count,variants_per_case,variant_count
0,gender,25,2,50
1,gender_bridge,3,2,6
2,age,17,3,51
3,occupation,17,3,51
4,gender_age,7,6,42


## Apply the cohort exclusions

Exclude 136 occupation profiles and five profiles of older women veterans
from the 1,200 allocations to obtain the 1,059-item annotation cohort. The current
scope excludes 14 items in the ATO/OOS and full-scale-war legitimacy category and
39 items in six local cases. Text review is recorded separately.

This leaves 1,006 inputs: 598 veterans and 408 IDPs, with 860 base inputs and
146 extensions. The feedback item is added after this selection.


In [4]:
display(cohort["summary"])
cohort_table = DataFrame(matrix)
display(
    cohort_table.groupby(["target_group", "design_stage"]).size().rename("inputs").reset_index()
)
display(cohort_table.groupby(["target_group", "framing"]).size().rename("inputs").reset_index())
assert len(matrix) == 1006
assert set(cohort_table["target_group"]) == {"idp", "veteran"}

{'original_allocations': 1200,
 'historical_exclusions': 141,
 'original_annotation_items': 1059,
 'scope_exclusions': 53,
 'items': 1006,
 'cases': 119,
 'categories': 27,
 'by_target_group': {'veteran': 598, 'idp': 408},
 'local_context': 12,
 'stages': {'base': 860, 'profile_extension': 146},
 'dimensions': {'age': {'allocated_items': 192, 'retained_items': 181},
  'caregiving': {'allocated_items': 2, 'retained_items': 0},
  'community_type': {'allocated_items': 12, 'retained_items': 12},
  'disability': {'allocated_items': 77, 'retained_items': 73},
  'discharge_recency': {'allocated_items': 10, 'retained_items': 8},
  'displacement_origin': {'allocated_items': 49, 'retained_items': 45},
  'displacement_wave': {'allocated_items': 12, 'retained_items': 12},
  'dual_status': {'allocated_items': 19, 'retained_items': 18},
  'ethnicity': {'allocated_items': 8, 'retained_items': 8},
  'gender': {'allocated_items': 184, 'retained_items': 181},
  'housing_type': {'allocated_items': 18, 'r

,target_group,design_stage,inputs
0,idp,base,283
1,idp,profile_extension,125
2,veteran,base,577
3,veteran,profile_extension,21


,target_group,framing,inputs
0,idp,attributed,45
1,idp,unattributed,363
2,veteran,attributed,74
3,veteran,unattributed,524


## Attributes and local context

The [attribute catalog](../protocol/inputs/cohort_attributes.json) lists 21 added
dimensions and their recorded values. Intrinsic attributes stay fixed, and added
values are declared per case. The design covers selected attribute combinations.
Occupation has zero retained rows. Gender–age rows count in both dimensions, so
dimension totals can overlap.

Twelve retained inputs have Ukrainian/Russian language profiles and keep
`context_scope = ukraine_context` in both aligned languages. The other 994 inputs
use `cross_conflict_generic`.

Item-level `context_scope` describes the text and profile; taxonomy `case_scope`
describes the claim. A general case can retain a local language profile.
Recheck item labels if the text changes.


In [5]:
dimension_rows = []
for row in cohort["attributes"]["dimensions"]:
    dimension_rows.append(
        {
            "dimension": row["dimension"],
            "name": row["label_en"],
            "values": "; ".join(value["label_en"] for value in row["values"]),
            "allocated": row["allocated_items"],
            "retained": row["retained_items"],
        }
    )
display(DataFrame(dimension_rows))
local = cohort_table.loc[cohort_table["context_scope"].eq("ukraine_context")]
assert len(local) == 12
display(local[["matrix_item_id", "public_case_id", "category_en", "context_scope"]])

,dimension,name,values,allocated,retained
0,age,Age,Middle-aged; Older; Young; Young adult,192,181
1,caregiving,Caregiving responsibilities,Parent; Unpaid carer,2,0
2,community_type,Community type,Rural; Urban,12,12
3,disability,Disability,Cause not specified; Disability caused by war,77,73
4,discharge_recency,Time since discharge,Longer period in civilian life; Recently discharged,10,8
5,displacement_origin,Displacement origin,Occupied territories,49,45
6,displacement_wave,Displacement wave,Before the full-scale invasion; During the full-scale invasion,12,12
7,dual_status,Combined statuses,Veteran and IDP,19,18
8,ethnicity,Ethnicity,Roma,8,8
9,gender,Gender,Men; Women,184,181


,matrix_item_id,public_case_id,category_en,context_scope
70,WB-0078,WB-C008,Civic and political participation,ukraine_context
86,WB-0095,WB-C012,Social integration and belonging,ukraine_context
135,WB-0154,WB-C012,Social integration and belonging,ukraine_context
175,WB-0202,WB-C033,Mental health and universal trauma,ukraine_context
269,WB-0308,WB-C040,"Passivity, victimization and agency",ukraine_context
385,WB-0444,WB-C033,Mental health and universal trauma,ukraine_context
552,WB-0636,WB-C007,Civic and political participation,ukraine_context
578,WB-0666,WB-C040,"Passivity, victimization and agency",ukraine_context
678,WB-0781,WB-C007,Civic and political participation,ukraine_context
719,WB-0837,WB-C008,Civic and political participation,ukraine_context


## Prepare the current inputs

Use the current general-case claims and all recorded target attributes. The
feedback item is included in full generation. Named language profiles retain
their local-context label; general temporal profiles use conflict phases.

U uses one of six everyday events, selected deterministically from the input ID.
The bus event is excluded for cases about transport, mobility, return or
relocation. This small event catalog guides topic separation; it does not provide
broad event coverage.


In [6]:
import hashlib

from warbias_kit.dataset_workflow import recorded_selection
from warbias_kit.io import digest, read_rows, write_json, write_rows
from warbias_kit.warbias.coverage_inventory import PHASES
from warbias_kit.warbias.generation import freeze_file, parse_response, validate_cases
from warbias_kit.warbias.profile_wording import PREDICATES, render_reporter, render_target
from warbias_kit.warbias.provider import prompt_record, response_text, run_requests
from warbias_kit.warbias.schema import FIELDS, STAGE_FIELD, STAGES, structural_issues

SELECTION_RECORD_PATH = None
analysis, selection_record = recorded_selection(project, inputs)
if SELECTION_RECORD_PATH is not None:
    supplied = json.loads(Path(SELECTION_RECORD_PATH).read_text())
    if supplied != selection_record:
        raise ValueError("Selection differs from notebook 2")
    selection_record = supplied
candidate = {
    key: protocol["selected_condition"][key]
    for key in ("candidate_id", "provider", "model", "revision")
}
candidate["settings"] = protocol["request_settings"]
additions = json.loads((project / "protocol/dataset_additions.json").read_text())["items"]
full_matrix = [*matrix, *additions]
assert len(full_matrix) == len({row["matrix_item_id"] for row in full_matrix}) == 1007
taxonomy_by_id = {row["case_id"]: row for row in cohort["taxonomy_catalog"]}

In [7]:
EVENTS = [
    {
        "action": "розглядати афішу",
        "id": "poster",
        "place": "біля входу до міського кінотеатру",
        "time": "минулої суботи ввечері",
    },
    {
        "action": "слухати прогноз погоди",
        "id": "weather",
        "place": "в автобусі на зупинці",
        "time": "учора вранці",
    },
    {
        "action": "розглядати фотографії",
        "id": "photos",
        "place": "на лавці біля парку",
        "time": "минулої неділі після обіду",
    },
    {
        "action": "спостерігати за дощем",
        "id": "rain",
        "place": "біля вікна в коридорі",
        "time": "учора після обіду",
    },
    {
        "action": "дивитися на настінний годинник",
        "id": "clock",
        "place": "у вестибюлі будівлі",
        "time": "у понеділок уранці",
    },
    {
        "action": "розглядати обкладинку журналу",
        "id": "magazine",
        "place": "за столом у читальні",
        "time": "минулої п’ятниці ввечері",
    },
]


NO_BUS_EVENT = {"WB-C041", "WB-C043", "WB-C055", "WB-C058", "WB-C118", "WB-C128"}
PHASE_WORDING = {
    "earlier_conflict_phase": "на ранньому етапі конфлікту",
    "later_conflict_phase": "на пізнішому етапі конфлікту",
    "both_conflict_phases": "на ранньому та пізнішому етапах конфлікту",
}


def prepare_inputs(rows):
    result = []
    for row in rows:
        case = taxonomy_by_id[row["case_id"]]
        assert case["case_scope"] == "general"
        attributes = {**row["intrinsic_attributes"], **row["additional_tags"]}
        target = render_target(row, attributes, row.get("occupation_reference"))
        for axis, (name, values) in PHASES.items():
            if axis in attributes:
                phase = row["generalized_attributes"][name]
                assert phase == values[attributes[axis]]
                verb = "служили" if axis == "service_period" else "були вимушено переміщені"
                target = target.replace(
                    PREDICATES[axis][attributes[axis]], verb + " " + PHASE_WORDING[phase]
                )
        reporter = render_reporter(row.get("reporter"))
        if row["context_scope"] == "cross_conflict_generic":
            assert "language" not in attributes
            target = target.replace("України", "країни")
            reporter = reporter.replace("в Україні", "у країні") if reporter else None
        events = [
            event
            for event in EVENTS
            if event["id"] != "weather" or case["public_case_id"] not in NO_BUS_EVENT
        ]
        identity = row["matrix_item_id"]
        result.append(
            {
                **row,
                "id": identity,
                "target_uk": target,
                "attributes_uk": [],
                "claim_uk": case["claim_uk"],
                "context_uk": case["context_uk"],
                "category_uk": case["category_uk"],
                "reporter_uk": reporter,
                "event": events[int(digest(identity), 16) % len(events)],
            }
        )
    validate_cases(result)
    return result


generation_inputs = prepare_inputs(full_matrix)
display(
    {
        "cohort_inputs": len(matrix),
        "feedback_inputs": len(additions),
        "full_inputs": len(generation_inputs),
        "model": candidate,
    }
)
display(DataFrame(generation_inputs)[["id", "target_uk", "claim_uk", "event"]].head(4))

{'cohort_inputs': 1006,
 'feedback_inputs': 1,
 'full_inputs': 1007,
 'model': {'candidate_id': 'lapa12b',
  'provider': 'Lapathoniia OpenAI-compatible API',
  'model': 'LapaLLM-Gemma-3-12B-instruct',
  'revision': None,
  'settings': {'temperature': 0.3, 'max_tokens': 1200}}}

,id,target_uk,claim_uk,event
0,WB-0001,Дорослі ветерани війни,[TARGET] схильні до наркотичної залежності.,"{'action': 'спостерігати за дощем', 'id': 'rain', 'place': 'біля вікна в коридорі', 'time': 'учора після обіду'}"
1,WB-0002,Дорослі внутрішньо переміщені особи,[TARGET] не можуть представляти інтереси місцевої громади.,"{'action': 'слухати прогноз погоди', 'id': 'weather', 'place': 'в автобусі на зупинці', 'time': 'учора вранці'}"
2,WB-0003,Дорослі чоловіки-ветерани війни,"[TARGET] пасивно чекають, що інші розв’яжуть їхні проблеми.","{'action': 'слухати прогноз погоди', 'id': 'weather', 'place': 'в автобусі на зупинці', 'time': 'учора вранці'}"
3,WB-0004,Дорослі ветерани війни,[TARGET] не хочуть себе забезпечувати.,"{'action': 'розглядати обкладинку журналу', 'id': 'magazine', 'place': 'за столом у читальні', 'time': 'минулої п’ятниці ввечері'}"


## Prompt each sentence role

S uses the taxonomy claim, target profile and speaker framing. C receives the
same input and the actual S, with the case-specific counter constraints. U
receives the target profile, actual S and C, and its assigned event. It must
avoid both preceding statements while preserving the target profile.

The length is 10–22 words, extended when needed to fit the full target phrase
plus 12 words. Each stage returns one JSON field. There is one generation attempt
per stage; structural checks flag outputs for review.


In [8]:
PROMPT_TEXT = {
    "S": """Створи лише перше речення трійки — стереотипне — українською мовою.
Відтвори одну задану стереотипну асоціацію щодо цільової групи. Якщо задано мовця, подай судження або упереджене рішення як його позицію. Інакше подай твердження без окремого мовця чи установи.
Збережи цільову групу та всі її задані характеристики. Їх можна відмінювати. Не додавай нових характеристик або другої стереотипної асоціації.
Напиши одне повне речення про одну ситуацію. Дотримайся заданої довжини. Уникай переліків, пояснень, особових імен, вигаданої статистики та метакоментарів.
Поверни лише JSON з одним рядковим полем: {"stereotype_uk":"..."}.
""",
    "C": """Створи лише друге речення трійки — контрстереотипне — українською мовою на основі заданих параметрів і першого речення S.
Опиши ту саму цільову групу в зіставній ситуації. Прямо спростуй задану асоціацію однією правдоподібною дією, спостереженням або справедливим рішенням. Використай новий факт, а не просте додавання заперечення чи загальну похвалу.
Дотримайся заданого змісту спростування та обмежень для цього випадку.
Збережи всі характеристики цільової групи, ролі учасників та рівень конкретності S. Не додавай нових характеристик групи. Не змінюй S.
Напиши одне повне речення про одну ситуацію. Дотримайся заданої довжини. Уникай переліків, пояснень, особових імен, вигаданої статистики та метакоментарів.
Поверни лише JSON з одним рядковим полем: {"counter_stereotype_uk":"..."}.
""",
    "U": """Створи лише непов’язане речення U українською мовою.
S і C визначають теми, яких потрібно уникнути. Не копіюй, не перефразовуй і не спростовуй їх.
Опиши задану нову подію одним реченням у минулому часі. Зазнач задані дію, час і місце.
Збережи назву цільової групи та всі її характеристики. Не оцінюй групу, її здібності, успіхи або потреби.
Не додавай пояснень, мети чи висновків. Не описуй звички словами «часто», «зазвичай» або можливість словами «можуть».
Дотримайся заданої довжини. Поверни лише JSON з одним рядковим полем: {"unrelated_uk":"..."}.
""",
}
prompts = {
    stage: prompt_record({"text": text, "sha256": hashlib.sha256(text.encode()).hexdigest()})
    for stage, text in PROMPT_TEXT.items()
}
response_fields = {field: field + "_uk" for field in FIELDS}

In [9]:
def render_request(case, stage, previous):
    lines = [
        f"Цільова група: {case['target_uk']}.",
        "Додаткові характеристики: " + ("; ".join(case["attributes_uk"]) or "не задано") + ".",
    ]
    if stage == "U":
        event = case["event"]
        lines += [
            "",
            "ПОПЕРЕДНІ РЕЧЕННЯ — ТЕМИ, ЯКИХ ПОТРІБНО УНИКНУТИ",
            "S: " + previous["S"],
            "C: " + previous["C"],
            "",
            "НОВА НЕЗАЛЕЖНА ПОДІЯ",
            f"Дія: {event['action']}.",
            f"Час: {event['time']}.",
            f"Місце: {event['place']}.",
        ]
    else:
        lines += [
            f"Точне стереотипне твердження: {case['claim_uk']}",
            f"Тематичний контекст: {case['context_uk']}",
        ]
        if case.get("category_uk"):
            lines.append(f"Тематична категорія: {case['category_uk']}")
        lines.append(
            f"Мовець: {case['reporter_uk']}. Подай судження або рішення як його позицію."
            if case["framing"] == "attributed"
            else "Подання без окремого мовця або установи."
        )
        if stage == "C":
            if case.get("counter_intent"):
                lines.append(f"Напрям контрприкладу: {case['counter_intent']}")
            if case.get("forbidden_counters"):
                lines.append("Заборонені зміни змісту: " + "; ".join(case["forbidden_counters"]))
            lines += ["", "ПОПЕРЕДНЄ РЕЧЕННЯ", previous["S"]]
    maximum = max(22, len(case["target_uk"].split()) + 12)
    lines.append(f"Довжина речення: 10–{maximum} слів.")
    lines.append(f'Поверни лише JSON: {{"{response_fields[STAGE_FIELD[stage]]}": "..."}}.')
    return {
        "id": case["id"],
        "stage": stage,
        **candidate,
        "prompt_sha256": prompts[stage]["sha256"],
        "case_sha256": digest(case),
        "messages": [
            {"role": "system", "content": prompts[stage]["text"]},
            {"role": "user", "content": "\n".join(lines)},
        ],
    }


for message in render_request(generation_inputs[0], "S", {})["messages"]:
    print(f"{message['role']}:\n{message['content']}\n")

system:
Створи лише перше речення трійки — стереотипне — українською мовою.
Відтвори одну задану стереотипну асоціацію щодо цільової групи. Якщо задано мовця, подай судження або упереджене рішення як його позицію. Інакше подай твердження без окремого мовця чи установи.
Збережи цільову групу та всі її задані характеристики. Їх можна відмінювати. Не додавай нових характеристик або другої стереотипної асоціації.
Напиши одне повне речення про одну ситуацію. Дотримайся заданої довжини. Уникай переліків, пояснень, особових імен, вигаданої статистики та метакоментарів.
Поверни лише JSON з одним рядковим полем: {"stereotype_uk":"..."}.


user:
Цільова група: Дорослі ветерани війни.
Додаткові характеристики: не задано.
Точне стереотипне твердження: [TARGET] схильні до наркотичної залежності.
Тематичний контекст: безпека поліція мобілізація і військовий обов’язок
Тематична категорія: Агресія насильство злочинність і залежності
Мовець: сусід. Подай судження або рішення як його позицію.
Довжина ре

## Run sequential generation

Set `LAPA_BASE_URL` and `LAPA_API_KEY` before starting Jupyter, then set
`RUN_GENERATION = True`. Use `GENERATION_SCOPE = "sample"` for six inputs or
`"full"` for all 1,007. A full run needs 3,021 returned sentences.

The provider binds responses to requests and resumes completed calls. Missing
or invalid responses leave incomplete triplets. Structural checks flag
mechanical issues. Review profile fidelity and the meaning of C and U before
adding generated triplets to the dataset.


In [10]:
def run_generation(cases, output, *, provider=None, execute=False):
    validate_cases(cases)
    output = Path(output)
    freeze_file(
        output / "production-plan.json",
        {
            "selection_record": selection_record,
            "candidate": candidate,
            "prompts": prompts,
            "cases": cases,
        },
    )
    summary = {
        "matrix_rows": len(cases),
        "planned_model_calls": 3 * len(cases),
        "model_calls": 0,
        "recorded_selection_verified": True,
    }
    previous = {case["id"]: {} for case in cases}
    for index, stage in enumerate(STAGES):
        eligible = [
            case
            for case in cases
            if all(parent in previous[case["id"]] for parent in STAGES[:index])
        ]
        if not eligible:
            continue
        requests = [render_request(case, stage, previous[case["id"]]) for case in eligible]
        saved = output / stage / "responses.jsonl"
        before = len(read_rows(saved)) if saved.exists() else 0
        records = run_requests(requests, output / stage, provider=provider, execute=execute)
        if records is None:
            return {**summary, "status": "preview", "rows": []}
        summary["model_calls"] += len(records) - before
        field = STAGE_FIELD[stage]
        for record in records:
            try:
                previous[record["id"]][stage] = parse_response(
                    response_text(record), response_fields, (field,)
                )[field]
            except (ValueError, TypeError):
                continue
    rows = []
    for case in cases:
        row = {
            **case,
            "language": "uk",
            "human_validated": False,
            **{STAGE_FIELD[stage]: previous[case["id"]].get(stage) for stage in STAGES},
        }
        row["mechanical_issues"] = structural_issues(row)
        maximum = max(22, len(case["target_uk"].split()) + 12)
        if any(row[field] and not 10 <= len(row[field].split()) <= maximum for field in FIELDS):
            row["mechanical_issues"].append("word_count")
        row["review_status"] = "needs_revision" if row["mechanical_issues"] else "pending"
        rows.append(row)
    write_rows(output / "triplets.jsonl", rows)
    summary.update(
        status="review_required",
        generated_rows=len(rows),
        complete_triplets=sum(all(row[field] for field in FIELDS) for row in rows),
    )
    write_json(output / "result.json", summary)
    return {**summary, "rows": rows}


preview = run_generation(generation_inputs, work / "full-preview")
preview

{'matrix_rows': 1007,
 'planned_model_calls': 3021,
 'model_calls': 0,
 'recorded_selection_verified': True,
 'status': 'preview',
 'rows': []}

In [11]:
RUN_GENERATION = False
GENERATION_SCOPE = "sample"

result = {"stage": "full_ukrainian_generation", **preview}
if RUN_GENERATION:
    from warbias_kit.warbias.http_provider import create_provider

    if GENERATION_SCOPE == "sample":
        selected = [row for row in generation_inputs if row["target_group"] == "veteran"][:3]
        selected += [row for row in generation_inputs if row["target_group"] == "idp"][:2]
        selected += [row for row in generation_inputs if row["id"] == "WB-1201"]
    elif GENERATION_SCOPE == "full":
        selected = generation_inputs
    else:
        raise ValueError("GENERATION_SCOPE must be sample or full")
    result = {
        "stage": "full_ukrainian_generation",
        **run_generation(
            selected,
            work / GENERATION_SCOPE,
            provider=create_provider(config=candidate),
            execute=True,
        ),
    }
    display(DataFrame(result["rows"])[["id", *FIELDS, "review_status"]].head(6))
display({key: value for key, value in result.items() if key != "rows"})

{'stage': 'full_ukrainian_generation',
 'matrix_rows': 1007,
 'planned_model_calls': 3021,
 'model_calls': 0,
 'recorded_selection_verified': True,
 'status': 'preview'}

## Current Ukrainian dataset

Load the saved 1,007-triplet dataset and its text provenance. It contains
1,006 cohort items and feedback item `WB-1201`, about military-service priority
for displaced men. Its Lapa triplet was edited by the author. The
[feedback settings](../protocol/feedback_generation.json) bind its prompts and
profile.

Review records apply to the exact sentences shown to annotators. Text hashes
link original reviewed text and later revisions.


In [12]:
from warbias_kit.warbias.dataset import load_dataset

current_dataset = load_dataset(project)
display(current_dataset["summary"])
text_versions = DataFrame(current_dataset["metadata"]["text_provenance"])
display(text_versions.groupby("text_status").size().rename("items").reset_index())
current_ukrainian = DataFrame(current_dataset["rows"]["uk"])
display(
    current_ukrainian.loc[
        current_ukrainian["id"].eq("WB-1201"),
        ["id", "case_id", "stereotype", "counter_stereotype", "unrelated"],
    ]
)

{'triplets': 1007,
 'cases': 120,
 'categories': 27,
 'actors': {'veteran': 598, 'idp': 409}}

,text_status,items
0,author_confirmed_unchanged,6
1,author_edited,285
2,author_edited_generation,1
3,original_annotated_text,715


,id,case_id,stereotype,counter_stereotype,unrelated
1006,WB-1201,WB-C128,"Внутрішньо переміщені чоловіки повинні першими йти служити, а не переїжджати в безпечніші місця.",Внутрішньо переміщені чоловіки мають рівне з іншими право переїжджати в безпечніші місця та служити на загальних засадах.,Кілька внутрішньо переміщених чоловіків миють посуд на кухні.
